# Flamingo 针对少样本的门控交叉注意力

DeepMind的Flamingo在其他人之前做了两件事。它表明一个模型可以处理任意图像、音频和文字的交叉。并且表明VLMs可以从Context中学到东西————给一个少样本的提示词，包含三个图像的标题的对，然后模型就可以给一个新的图像写标题，不需要任何梯度相关的计算。机制是：门控交叉注意力，插入到冻结的LLM层之间，附上一个可学习的tanh门控，从零开始，等价于模型的文本能力在开始的时候被完全保留。

## 问题描述

BLIP-2将32个视觉Token注入到LLM的输入层。这对每条提示词一个图像的场景有效，但是如果你想同时喂图像和文本的交叉序列呢。LLM的自注意力中输入图片和文本Token在同一个流上，问题繁琐在于哪些位置可以看到哪些图像。

Flamingo 的答案是：不要修改LLM的输入流，在已有的LLM块中插入额外的交叉注意力层。文本Token仍遵循LLM的因果自注意力。在每隔一部分LLM块之后，文本Token通过门控层同样的交叉关注图像特征。门控表示在第零步，新的这些层就跟没有操作一样，模型表现跟预训练的结果一样。训练开始后，门控生效，视觉信息开始流动。

Flamigo回答的第二个问题是：怎么处理单条提示词中不定数量的图像。一个感知器重采样机————一个小的交叉注意力模块，以任意数量的Patches产生固定数量的视觉潜空间Token。然后LLM的交叉注意力层处理的形状相同，而不用关注你输入了多少张图片。

## 基本概念

### 感知器重采样

对于提示词中的每张图片，ViT会产生N张Patch Token。感知器重采样机有固定K个可学习的向量。每个采样器分成两个阶段。
- 交叉注意力：K个潜变量关注N个Patch Token (Q从潜变量来，K/V从Patch Token来)
- 潜变量自注意力和前馈网络

经过六层重采样块后，输出K=64个dim为1024的视觉Token，不管原始ViT产生了多少个Patch。

对于视频来说，采样器在每一帧完成，产生64个潜变量，加上一个临时的位置编码让模型分辨时间，整个视屏就变成了T*64个视觉Token。

### 门控交叉注意力

冻结的LLM每隔M层（Flamingo是4层），插入一个新的门控交叉注意力块。
```
x_after_llm_block = llm_block(x_before)
cross = cross_attn(x_after, resampler_output)
gated = tanh(alph) * cross + x_after
x_before_next_block = gated
```
`alpha` 是一个可学习的标量，初始化为0。因为`tanh(0) = 0`以及残差注意力，所以初始时门控分枝不做贡献，当`alpha`逐渐离开0，交叉注意力贡献平稳上升。

这就是Flamingo中做的最重要的一个设计：视觉条件是附加的、门控的初始化为零的。

### 交叉输入的掩码交叉注意力

如果提示词像这样"ImageA CaptionA ImageB CaptionB ... " 每个文本Token应该只能看到序列中的上一个图像。交叉注意力掩码强制：在位置t处的文本Token只关注图像索引i<i_t的图片，i_t表示t以前最近的图像。看见以前所有的图像和只看见以前最近的图像都是可行的方案。Flamingo的选择是前者。

### 上下文少样本学习

一个Flamingo的提示词长这样：
```
<Image1> A cat. <Image2> A dog. <Image3> A
```
模型学会这个模式，然后输出Image3的分类。没有梯度计算过程。门控交叉注意力给冻结的LLM带来了上下文学习能力，这是论文的核心买点。

### 对比BLIP-2

||BLIP-2|Flamingo|
|---|---|---|
|视觉桥梁|Q-Former在输入注入|门控交叉注意力每M层注意|
|视觉Token数|每张图片32个|每个交叉注意力层每张图片64个|
|Forzen LLM|Yes|Yes|
|上下文少样本|弱|强|
|交叉输入|无原生支持|支持|

# 开始编码

教学积木：小尺寸复现 Flamingo 三个核心——Perceiver Resampler、门控交叉注意力、交错图文掩码。


## 1. Perceiver Resampler

任意 N 个 ViT patch → 固定 K 个视觉 latent（可学习 query 做交叉注意，再自注意+FFN）。


In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyFlamingoConfig:
    """小尺寸配置，方便打印 shape。"""
    vit_dim: int = 48
    dim: int = 64                 # resampler / 门控层隐维
    llm_dim: int = 64            # 玩具 LLM 与 dim 对齐，省一层投影
    n_heads: int = 4
    num_latents: int = 8         # 真 Flamingo K=64
    resampler_layers: int = 2    # 真模型约 6
    num_patches: int = 16
    vocab_size: int = 40
    n_llm_blocks: int = 4
    insert_every: int = 2        # 每隔 M 个 LLM block 插门控交叉注意（真 Flamingo M=4）
    max_images: int = 3


class CrossAttention(nn.Module):
    def __init__(self, dim: int, context_dim: int, n_heads: int):
        super().__init__()
        self.ctx_proj = nn.Linear(context_dim, dim) if context_dim != dim else nn.Identity()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(
        self,
        x: torch.Tensor,
        context: torch.Tensor,
        attn_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        kv = self.ctx_proj(context)
        h, _ = self.attn(x, kv, kv, attn_mask=attn_mask, need_weights=False)
        return self.norm(x + h)


class SelfAttentionFFN(nn.Module):
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )
        self.n1 = nn.LayerNorm(dim)
        self.n2 = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h, _ = self.attn(x, x, x, need_weights=False)
        x = self.n1(x + h)
        return self.n2(x + self.ffn(x))


class PerceiverResampler(nn.Module):
    """
    每张图：N patch -> K latent。
    latents 是可学习参数（训练更新，推理权重固定，输出随图变）。
    """

    def __init__(self, cfg: TinyFlamingoConfig):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(1, cfg.num_latents, cfg.dim) * 0.02)
        self.layers = nn.ModuleList()
        for _ in range(cfg.resampler_layers):
            self.layers.append(
                nn.ModuleDict(
                    {
                        "cross": CrossAttention(cfg.dim, cfg.vit_dim, cfg.n_heads),
                        "self": SelfAttentionFFN(cfg.dim, cfg.n_heads),
                    }
                )
            )

    def forward_one(self, patches: torch.Tensor) -> torch.Tensor:
        """patches: (B, N, vit_dim) -> (B, K, dim)"""
        B = patches.size(0)
        x = self.latents.expand(B, -1, -1)
        for layer in self.layers:
            x = layer["cross"](x, patches)
            x = layer["self"](x)
        return x

    def forward_many(self, images_patches: torch.Tensor) -> torch.Tensor:
        """
        images_patches: (B, T, N, vit_dim)  T 张图
        return: (B, T, K, dim)
        """
        B, T, N, D = images_patches.shape
        flat = images_patches.reshape(B * T, N, D)
        out = self.forward_one(flat)
        K = out.size(1)
        return out.view(B, T, K, -1)


print("PerceiverResampler ready")


## 2. 门控交叉注意力

`gated = x + tanh(α) * cross_attn(x, media)`，α 初始化为 0 → 起步等价于纯文本 LLM。


In [ ]:
class GatedCrossAttention(nn.Module):
    """插入冻结 LLM 层之间的视觉条件分支。"""

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.cross = CrossAttention(dim, dim, n_heads)
        # 标量门：tanh(0)=0，残差旁路一开始不贡献
        self.alpha = nn.Parameter(torch.tensor(0.0))

    def forward(
        self,
        x: torch.Tensor,
        media: torch.Tensor,
        attn_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # x: (B, L_text, D)
        # media: (B, T*K, D) 或已展平的视觉 token
        cross = self.cross(x, media, attn_mask=attn_mask)
        gate = torch.tanh(self.alpha)
        return x + gate * cross


# 验证初始门控 ≈ 恒等
_cfg = TinyFlamingoConfig()
_g = GatedCrossAttention(_cfg.dim, _cfg.n_heads)
_x = torch.randn(2, 5, _cfg.dim)
_m = torch.randn(2, 8, _cfg.dim)
with torch.no_grad():
    _y = _g(_x, _m)
print(f"alpha={_g.alpha.item():.1f}, tanh(alpha)={torch.tanh(_g.alpha).item():.1f}")
print(f"max|y-x| at init = {(_y - _x).abs().max().item():.2e}  (应接近 0)")


## 3. 交错图文：文本只看「之前的图像」

构造 cross-attn mask：位置 t 的文本可见所有 i < i_t 的图像 latents（Flamingo：先前全部图像）。


In [ ]:
def build_media_mask(
    text_len: int,
    num_images: int,
    latents_per_image: int,
    text_to_image_idx: list[int],
    device,
) -> torch.Tensor:
    """
    text_to_image_idx[t] = 该文本位置「之前最近」的图像下标；
    -1 表示此前还没有任何图像。

    返回 attn_mask: (L_text, T*K)，对 MultiheadAttention 为 float mask：
    0=可见，-inf=屏蔽。
    Flamingo 选择：可见所有 i <= text_to_image_idx[t] 的图像（先前全部）。
    """
    Tk = num_images * latents_per_image
    mask = torch.full((text_len, Tk), float("-inf"), device=device)
    for t, img_i in enumerate(text_to_image_idx):
        if img_i < 0:
            continue
        # 可见图像 0..img_i（含当前最近一张）
        end = (img_i + 1) * latents_per_image
        mask[t, :end] = 0.0
    return mask


def demo_mask():
    # 交错示意: [img0] tok tok [img1] tok tok tok
    # 文本 5 个位置，对应最近图像: 0,0,1,1,1
    text_to_image = [0, 0, 1, 1, 1]
    m = build_media_mask(
        text_len=5,
        num_images=2,
        latents_per_image=3,
        text_to_image_idx=text_to_image,
        device="cpu",
    )
    print("rows=text positions, cols=image latents [img0|img0|img0|img1|img1|img1]")
    print(m)
    print("前两行只能看 img0；后三行可看 img0+img1")


demo_mask()


## 4. 拼装 TinyFlamingo + 少样本交错冒烟测试


In [ ]:
class FrozenToyLLMBlock(nn.Module):
    """冻结因果自注意力块（玩具）。"""

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))
        self.n1 = nn.LayerNorm(dim)
        self.n2 = nn.LayerNorm(dim)
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.size(1)
        # 因果 mask
        causal = torch.triu(torch.full((L, L), float("-inf"), device=x.device), diagonal=1)
        h, _ = self.attn(x, x, x, attn_mask=causal, need_weights=False)
        x = self.n1(x + h)
        return self.n2(x + self.ffn(x))


class TinyFlamingo(nn.Module):
    """
    文本只走冻结 LLM；每隔 insert_every 层插入可训练门控交叉注意。
    视觉：PerceiverResampler 把每张图压成 K latent。
    """

    def __init__(self, cfg: TinyFlamingoConfig):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab_size, cfg.llm_dim)
        self.resampler = PerceiverResampler(cfg)
        self.llm_blocks = nn.ModuleList(
            [FrozenToyLLMBlock(cfg.llm_dim, cfg.n_heads) for _ in range(cfg.n_llm_blocks)]
        )
        self.gated_layers = nn.ModuleDict()
        for i in range(cfg.n_llm_blocks):
            if (i + 1) % cfg.insert_every == 0:
                self.gated_layers[str(i)] = GatedCrossAttention(cfg.llm_dim, cfg.n_heads)
        self.head = nn.Linear(cfg.llm_dim, cfg.vocab_size)
        for p in self.tok.parameters():
            p.requires_grad = False
        for p in self.head.parameters():
            p.requires_grad = False

    def forward(
        self,
        text_ids: torch.Tensor,
        images_patches: torch.Tensor,
        text_to_image_idx: list[int],
    ) -> torch.Tensor:
        """
        text_ids: (B, L)
        images_patches: (B, T, N, vit_dim)
        text_to_image_idx: 长度 L，每个文本位置对应最近图像下标
        """
        B, T, N, _ = images_patches.shape
        media = self.resampler.forward_many(images_patches)  # (B, T, K, D)
        K = media.size(2)
        media_flat = media.reshape(B, T * K, -1)

        mask = build_media_mask(
            text_len=text_ids.size(1),
            num_images=T,
            latents_per_image=K,
            text_to_image_idx=text_to_image_idx,
            device=text_ids.device,
        )

        x = self.tok(text_ids)
        for i, block in enumerate(self.llm_blocks):
            x = block(x)
            if str(i) in self.gated_layers:
                x = self.gated_layers[str(i)](x, media_flat, attn_mask=mask)
        return self.head(x)


def smoke_test():
    torch.manual_seed(0)
    cfg = TinyFlamingoConfig()
    model = TinyFlamingo(cfg)

    B, T, L = 2, 3, 6
    # 少样本交错：img0 后 2 token，img1 后 2 token，img2 后 2 token
    text_to_image = [0, 0, 1, 1, 2, 2]
    text_ids = torch.randint(1, cfg.vocab_size, (B, L))
    images = torch.randn(B, T, cfg.num_patches, cfg.vit_dim)

    print("=== shapes ===")
    media = model.resampler.forward_many(images)
    print(f"images patches: {tuple(images.shape)}")
    print(f"resampled media: {tuple(media.shape)}  # (B, T, K, D) 与 N 无关")

    logits = model(text_ids, images, text_to_image)
    print(f"logits: {tuple(logits.shape)}")

    # 只应训练 resampler + gated（alpha 等）
    trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    train_n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\ntrainable params: {train_n}  frozen: {frozen}")
    print("trainable modules (sample):", [n for n, _ in trainable[:8]], "...")

    # 门控初始仍接近恒等路径（相对「无视觉」）
    alphas = [float(torch.tanh(g.alpha).detach()) for g in model.gated_layers.values()]
    print(f"tanh(alpha) per gated layer at init: {alphas}")

    # 一步玩具语言建模 loss，确认视觉分支可反传
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    opt.zero_grad()
    logits = model(text_ids, images, text_to_image)
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, cfg.vocab_size), text_ids[:, 1:].reshape(-1))
    loss.backward()
    opt.step()
    alphas_after = [float(torch.tanh(g.alpha).detach()) for g in model.gated_layers.values()]
    print(f"loss={loss.item():.4f}")
    print(f"tanh(alpha) after 1 step: {alphas_after}  (应开始离开 0)")
    print("SMOKE TEST OK")


smoke_test()
